In [1]:
!pip install pdfplumber pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 548.0 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 716.0 kB/s eta 0:00:00a 0:00:01


In [1]:
import pdfplumber
import pandas as pd

pdf_path = "../data/Band-Descriptors-Task-2.pdf"

tables = []
with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        table = page.extract_table()
        if table:
            tables.append(table)

# chuyển sang DataFrame
df = pd.DataFrame(tables[0][1:], columns=tables[0][0])
df

,Band,Task response,Coherence and cohesion,Lexical resource,Grammatical range and accuracy
0,9,• fully addresses all parts of the task\n• pre...,• uses cohesion in such a way that it attracts...,• uses a wide range of vocabulary with very na...,• uses a wide range of structures with full fl...
1,8,• sufficiently addresses all parts of the task...,• sequences information and ideas logically\n•...,• uses a wide range of vocabulary fluently and...,• uses a wide range of structures\n• the major...
2,7,• addresses all parts of the task\n• presents ...,• logically organises information and ideas; t...,• uses a sufficient range of vocabulary to all...,• uses a variety of complex structures\n• prod...
3,6,• addresses all parts of the task although som...,• arranges information and ideas coherently an...,• uses an adequate range of vocabulary for the...,• uses a mix of simple and complex sentence fo...
4,5,• addresses the task only partially; the forma...,• presents information with some organisation ...,"• uses a limited range of vocabulary, but this...",• uses only a limited range of structures\n• a...
5,4,• responds to the task only in a minimal way o...,• presents information and ideas but these are...,• uses only basic vocabulary which may be used...,• uses only a very limited range of structures...
6,3,• does not adequately address any part of the ...,• does not organise ideas logically\n• may use...,• uses only a very limited range of words and\...,• attempts sentence forms but errors in gramma...
7,2,• barely responds to the task\n• does not expr...,• has very little control of organisational fe...,• uses an extremely limited range of vocabular...,• cannot use sentence forms except in memorise...
8,1,• answer is completely unrelated to the task,• fails to communicate any message,• can only use a few isolated words,• cannot use sentence forms at all
9,0,• does not attend\n• does not attempt the task...,None,None,None


In [2]:
criterion_map = {
    "Task response": "TR",
    "Coherence and cohesion": "CC",
    "Lexical resource": "LR",
    "Grammatical range and accuracy": "GRA"
}

In [3]:
import re

def clean_cell_text(text):
    if text is None:
        return ""
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [4]:
import json

records = []

for _, row in df.iterrows():
    band = float(row["Band"])

    for col, crit_code in criterion_map.items():
        cell_text = clean_cell_text(row[col])

        record = {
            "id": f"descriptor_{crit_code}_{int(band)}",
            "source_type": "descriptor",
            "criterion": crit_code,
            "band": band,
            "text": f"{col} band {band}:\n{cell_text}"
        }

        records.append(record)

In [5]:
with open("band_descriptors_task2.jsonl", "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

In [6]:
df_chunks = pd.DataFrame(records)
df_chunks.to_csv("band_descriptors_task2.csv", index=False)